# 01 — Data Prep

Pull 150 GSM8K problems, perturb numeric values, store ground truth.

In [ ]:
import re
import json
import random
from datasets import load_dataset

random.seed(42)
dataset = load_dataset("gsm8k", "main", split="test")
print(f"GSM8K test set size: {len(dataset)}")

In [ ]:
def perturb_value(num_str: str) -> str:
    """Scale a number by 0.5-1.5x, keeping int vs float type."""
    is_int = '.' not in num_str
    num = int(num_str) if is_int else float(num_str)
    factor = 0.5 + random.random() * 1.0  # [0.5, 1.5]
    new_num = num * factor
    if is_int:
        new_num = int(round(new_num))
    else:
        new_num = round(new_num, 2)
    return str(new_num)


def recompute_last_expression(text: str) -> float | None:
    """Find the last arithmetic expression in text and evaluate it."""
    matches = list(re.finditer(
        r'(\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)',
        text
    ))
    if not matches:
        return None
    a_str, op, b_str = matches[-1].groups()
    a, b = float(a_str), float(b_str)
    if op == '+': return a + b
    elif op == '-': return a - b
    elif op == '*': return a * b
    elif op == '/': return a / b if b != 0 else None
    return None


def perturb_problem(question: str, answer_chain: str, seed: int) -> tuple[str, float]:
    """Return (perturbed_question, new_correct_answer)."""
    random.seed(seed)
    # Extract unique numbers from the question (inputs only)
    numbers_in_q = list(dict.fromkeys(
        m.group(0) for m in re.finditer(r'\d+(?:\.\d+)?', question)
    ))
    # Build old->new mapping
    mapping = {n: perturb_value(n) for n in numbers_in_q}
    # Sort longest-first so we don't replace parts of longer numbers
    sorted_nums = sorted(numbers_in_q, key=len, reverse=True)
    # Replace in question
    new_q = question
    for old, new in [(n, mapping[n]) for n in sorted_nums]:
        new_q = re.sub(rf'\b{re.escape(old)}\b', new, new_q)
    # Replace same numbers in the answer chain
    new_chain = answer_chain
    for old, new in [(n, mapping[n]) for n in sorted_nums]:
        new_chain = re.sub(rf'\b{re.escape(old)}\b', new, new_chain)
    # Recompute the answer from the last arithmetic expression
    new_answer = recompute_last_expression(new_chain)
    if new_answer is None:
        # Fallback: try to extract and perturb the raw final answer
        m = re.search(r'####\s*(-?[\d.]+)', answer_chain)
        if m:
            orig_ans = m.group(1)
            if orig_ans in mapping:
                new_answer = float(mapping[orig_ans])
            else:
                new_answer = float(orig_ans)
        else:
            new_answer = 0.0
    # Round answer to avoid float noise
    if isinstance(new_answer, float):
        new_answer = round(new_answer, 2)
        if new_answer == int(new_answer):
            new_answer = int(new_answer)
    else:
        new_answer = int(new_answer) if new_answer == int(new_answer) else new_answer
    return new_q, new_answer

In [ ]:
problems = []
for i, example in enumerate(dataset):
    if i >= 150:
        break
    q, ans = perturb_problem(example["question"], example["answer"], seed=i)
    problems.append({"problem_text": q, "correct_answer": ans})

with open("data/perturbed_problems.json", "w") as f:
    json.dump(problems, f, indent=2)
print(f"Saved {len(problems)} perturbed problems to data/perturbed_problems.json")

In [ ]:
# Quick validation
answers = [p["correct_answer"] for p in problems]
print(f"Unique answers: {len(set(answers))} / {len(answers)}")
print(f"Answer range: {min(answers)} to {max(answers)}")